In [0]:
dbutils.import_notebook('api_servicenow')
dbutils.import_notebook('utils')

In [0]:
import json
from utils import app_log
from api_servicenow import request_create_in

In [0]:
def insert_table_control_in(id_processamento, status):
    spark.sql(f"INSERT INTO logs.controle_in (id_processamento, status) VALUES ({id_processamento}, {status})")

In [0]:
def main():
    #Obter registros para processamento
    df_erros = spark.sql("SELECT * FROM logs.controle_processamento AS CP LEFT ANTI JOIN logs.controle_in AS CI ON CP.id = CI.id_processamento WHERE CP.status = 'E'").toPandas()

    app_log('info',f'Quantidade de registro para criação de IN: {df_erros.shape[0]}')

    for _, row in df_erros.iterrows():
        try:
            app_log('info',f'Criando o IN para o processamento da camada: {row.camada_processamento} do dia: {row.data_processamento}')
            body = {
                'description':f'Erro ao processar a camada {row.camada_processamento}\nError: {row.erros}\nData do processamento: {row.data_processamento}',
                'urgency': 3,
                'priority': 3,
                'category': 'Software',
                'short_description': f'Error no processamento da camada {row.camada_processamento}'
            }
            status_code = request_create_in(body)
            if status_code != 201:
                insert_table_control_in(row.id, 'N/C')
                return
            insert_table_control_in(row.id, 'C')
            app_log('info', 'IN criado com sucesso.')
        except Exception as e:
            app_log('error', f'Aconteceu um erro ao criar o IN.\nError: {e}')

In [0]:
if __name__ == '__main__':
    main()